In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go
from dash import Dash, dcc, html, dash_table
from dash.dependencies import Input, Output
from jupyter_dash import JupyterDash


# 1️ CARGAR DATOS PRINCIPALES

df = pd.read_csv(r"C:\Users\JuanFelipeMorenoFigu\Documents\u\5to semestre\ciencia de datos II\Proyecto\Delitos_Depa_Colombia.csv")

# Limpieza y transformación
df.columns = df.columns.str.strip()
df['FECHA HECHO'] = pd.to_datetime(df['FECHA HECHO'], errors='coerce')
df['AÑO'] = df['FECHA HECHO'].dt.year
df['CANTIDAD'] = pd.to_numeric(df['CANTIDAD'], errors='coerce').fillna(0)

# Nombres de columnas relevantes
col_delito = 'TIPO_DELITO'
col_departamento = 'DEPARTAMENTO'
col_genero = 'GENERO'
col_edad = 'AGRUPA EDAD PERSONA'


# 2️ INICIALIZAR LA APP


app = JupyterDash(__name__)
app.title = "Delitos Colombia 2023-2025"


# 3️ LAYOUTS DE CADA PESTAÑA


# --- TAB 1: Comparar por Departamento ---
tab1_layout = html.Div([
    html.H3("Comparación de delitos por departamento"),
    html.Div([
        html.Label("Tipo de delito"),
        dcc.Dropdown(
            id='delito-selector',
            options=[{'label': i, 'value': i} for i in sorted(df[col_delito].dropna().unique())],
            value=sorted(df[col_delito].dropna().unique())[0]
        ),
        html.Label("Género"),
        dcc.Dropdown(
            id='genero-selector',
            options=[{'label': 'TODOS', 'value': 'TODOS'}] +
                    [{'label': i, 'value': i} for i in sorted(df[col_genero].dropna().unique())],
            value='TODOS'
        ),
        html.Label("Edad"),
        dcc.Dropdown(
            id='edad-selector',
            options=[{'label': 'TODOS', 'value': 'TODOS'}] +
                    [{'label': i, 'value': i} for i in sorted(df[col_edad].dropna().unique())],
            value='TODOS'
        )
    ], style={'width': '35%', 'display': 'inline-block', 'verticalAlign': 'top'}),

    dcc.Graph(id='grafico-departamento'),
    html.H3("Tabla resumen por departamento"),
    dash_table.DataTable(
        id='tabla-departamento',
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left'},
        export_format='csv',
        page_size=15
    )
])

# --- TAB 2: Comparar por Tipo de Delito ---

tab2_layout = html.Div([
    html.H3("Comparación de delitos dentro del departamento"),
    html.Label("Departamento"),
    dcc.Dropdown(
        id='departamento-selector',
        options=[{'label': i, 'value': i} for i in sorted(df[col_departamento].dropna().unique())],
        value=sorted(df[col_departamento].dropna().unique())[0]
    ),
    dcc.Graph(id='grafico-delito'),
    dash_table.DataTable(id='tabla-delito', export_format='csv', page_size=15)
])

# --- TAB 3: Diferencias Globales ---

df_global = pd.read_csv(r"C:\Users\JuanFelipeMorenoFigu\Documents\u\5to semestre\ciencia de datos II\Proyecto\Comparaciondelitosporaño.csv")
df_global.columns = df_global.columns.str.strip()
df_global['Dif_2024_2023'] = df_global['AÑO_2024'] - df_global['AÑO_2023']
df_global['Dif_2025_2024'] = df_global['AÑO_2025'] - df_global['AÑO_2024']
df_global['Dif_2025_2023'] = df_global['AÑO_2025'] - df_global['AÑO_2023']
años = ["AÑO_2023", "AÑO_2024", "AÑO_2025"]

tab3_layout = html.Div([
    html.H3("Diferencias de delitos entre años"),
    dcc.Dropdown(
        id='selector-delito',
        options=[{'label': delito, 'value': delito} for delito in df_global['TIPO_DELITO']],
        value=df_global['TIPO_DELITO'].iloc[0],
        clearable=False
    ),
    dcc.Graph(id='grafico-global'),
    html.Div(id='tabla-global')
])

# --- TAB 4: Diferencias Delictivas por Año ---

df_melted = df_global.melt(
    id_vars=["TIPO_DELITO"],
    value_vars=["AÑO_2023", "AÑO_2024", "AÑO_2025"],
    var_name="AÑO",
    value_name="CANTIDAD"
)
df_melted["AÑO"] = df_melted["AÑO"].str.extract(r'(\d{4})')

fig_tab4 = px.bar(
    df_melted,
    x="TIPO_DELITO",
    y="CANTIDAD",
    color="AÑO",
    barmode="group",
    text="CANTIDAD",
    title="Diferencias delictivas por año",
    color_discrete_map={"2023": "dodgerblue", "2024": "tomato", "2025": "green"}
)
fig_tab4.update_layout(
    xaxis_title="Tipo de delito",
    yaxis_title="Cantidad de casos",
    xaxis_tickangle=-45,
    legend_title="Año"
)
fig_tab4.update_traces(textposition="outside")

tab4_layout = html.Div([
    html.H3("Diferencias delictivas por año"),
    dcc.Graph(
        figure=fig_tab4,
        id='grafico-tab4'
    )
])


# 4️ ALLBACKS POR PESTAÑA


# TAB 1: Comparar por Departamento
@app.callback(
    [Output('grafico-departamento', 'figure'),
     Output('tabla-departamento', 'data'),
     Output('tabla-departamento', 'columns')],
    [Input('delito-selector', 'value'),
     Input('genero-selector', 'value'),
     Input('edad-selector', 'value')]
)
def actualizar_por_departamento(tipo, genero, edad):
    años_validos = [2023, 2024, 2025]
    df_filtrado = df[(df['AÑO'].isin(años_validos)) & (df[col_delito] == tipo)]
    if genero != 'TODOS':
        df_filtrado = df_filtrado[df[col_genero] == genero]
    if edad != 'TODOS':
        df_filtrado = df_filtrado[df[col_edad] == edad]

    grupo = df_filtrado.groupby(['DEPARTAMENTO', 'AÑO'])['CANTIDAD'].sum().unstack(fill_value=0).reset_index()
    grupo.columns = ['DEPARTAMENTO', 'AÑO_2023', 'AÑO_2024', 'AÑO_2025']
    grupo['Dif_23_24'] = grupo['AÑO_2024'] - grupo['AÑO_2023']
    grupo['Dif_24_25'] = grupo['AÑO_2025'] - grupo['AÑO_2024']
    grupo['Tendencia'] = grupo['Dif_24_25'].apply(lambda x: 'AUMENTO' if x > 0 else 'DISMINUCIÓN' if x < 0 else 'SIN CAMBIO')

    fig = px.bar(grupo, x='DEPARTAMENTO',
                 y=['AÑO_2023', 'AÑO_2024', 'AÑO_2025'],
                 barmode='group',
                 title=f"{tipo} por Departamento",
                 labels={'value': 'Casos', 'variable': 'Año'})
    fig.update_layout(xaxis_tickangle=-45)

    columnas = [{'name': c, 'id': c} for c in grupo.columns]
    return fig, grupo.to_dict('records'), columnas


# TAB 2: Comparar por Tipo de Delito
@app.callback(
    [Output('grafico-delito', 'figure'),
     Output('tabla-delito', 'data'),
     Output('tabla-delito', 'columns')],
    [Input('departamento-selector', 'value')]
)
def actualizar_por_delito(departamento):
    años_validos = [2023, 2024, 2025]
    df_filtrado = df[(df['AÑO'].isin(años_validos)) & (df[col_departamento] == departamento)]
    grupo = df_filtrado.groupby([col_delito, 'AÑO'])['CANTIDAD'].sum().unstack(fill_value=0).reset_index()
    grupo.columns = ['TIPO_DELITO', 'AÑO_2023', 'AÑO_2024', 'AÑO_2025']
    grupo['Dif_23_24'] = grupo['AÑO_2024'] - grupo['AÑO_2023']
    grupo['Dif_24_25'] = grupo['AÑO_2025'] - grupo['AÑO_2024']

    fig = px.bar(grupo, x='TIPO_DELITO',
                 y=['AÑO_2023', 'AÑO_2024', 'AÑO_2025'],
                 barmode='group',
                 title=f"Delitos en {departamento} por tipo")
    fig.update_layout(xaxis_tickangle=-45)
    columnas = [{'name': c, 'id': c} for c in grupo.columns]
    return fig, grupo.to_dict('records'), columnas


# TAB 3: Diferencias Globales
@app.callback(
    [Output('grafico-global', 'figure'),
     Output('tabla-global', 'children')],
    [Input('selector-delito', 'value')]
)
def actualizar_global(delito):
    fila = df_global[df_global['TIPO_DELITO'] == delito].iloc[0]
    valores = [fila[año] for año in años]

    fig = go.Figure(data=[go.Bar(x=años, y=valores, text=[f"{v:.0f}" for v in valores],
                                 textposition='outside', marker_color='cornflowerblue')])
    fig.update_layout(title=f"Evolución de {delito}", yaxis_title="Casos", xaxis_title="Año")

    tabla = html.Table([
        html.Thead(html.Tr([html.Th("Comparación"), html.Th("Diferencia")])),
        html.Tbody([
            html.Tr([html.Td("2024 vs 2023"), html.Td(f"{fila['Dif_2024_2023']:.0f}")]),
            html.Tr([html.Td("2025 vs 2024"), html.Td(f"{fila['Dif_2025_2024']:.0f}")]),
            html.Tr([html.Td("2025 vs 2023"), html.Td(f"{fila['Dif_2025_2023']:.0f}")])
        ])
    ], style={'width': '40%', 'margin': 'auto', 'border': '1px solid #ccc', 'marginTop': '20px'})

    return fig, tabla


# 5️ LAYOUT PRINCIPAL CON TODAS LAS PESTAÑAS


app.layout = html.Div([
    html.Div([
        html.Img(
            src="/assets/logo_compensar.png",
            style={'width': '200px', 'display': 'block', 'margin': '0 auto'}
        ),
        html.H1("Delitos en Colombia (2023-2025)",
                style={'textAlign': 'center', 'color': '#222'})
    ]),
    
    dcc.Tabs(id='tabs', value='tab1', children=[
        dcc.Tab(label='1️ Comparar por Departamento', value='tab1'),
        dcc.Tab(label='2️ Comparar por Tipo de Delito', value='tab2'),
        dcc.Tab(label='3️ Diferencias entre delictos por Año', value='tab3'),
        dcc.Tab(label='4️ Diferencia general por Año', value='tab4'),
    ]),
    html.Div(id='tabs-content')
])


# 6️ CALLBACK PRINCIPAL DE TABS


@app.callback(Output('tabs-content', 'children'),
              Input('tabs', 'value'))
def mostrar_tab(tab):
    if tab == 'tab1':
        return tab1_layout
    elif tab == 'tab2':
        return tab2_layout
    elif tab == 'tab3':
        return tab3_layout
    elif tab == 'tab4':
        return tab4_layout


# 7️ EJECUTAR APP EN JUPYTER

app.run(jupyter_mode="tab", port=8051)



C:\Users\JuanFelipeMorenoFigu\AppData\Local\Temp\ipykernel_16008\1726476486.py:11: DtypeWarning: Columns (0: TIPO_DELITO, 1: ARMAS MEDIOS, 2: DEPARTAMENTO, 3: MUNICIPIO, 4: FECHA HECHO, 5: GENERO, 6: AGRUPA EDAD PERSONA) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\JuanFelipeMorenoFigu\Documents\u\5to semestre\ciencia de datos II\Proyecto\Delitos_Depa_Colombia.csv")
C:\Users\JuanFelipeMorenoFigu\AppData\Roaming\Python\Python312\site-packages\jupyter_dash\jupyter_app.py:103: UserWarning: JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.
  super(JupyterDash, self).__init__(name=name, **kwargs)


Dash app running on http://127.0.0.1:8051/


<IPython.core.display.Javascript object>

C:\Users\JuanFelipeMorenoFigu\AppData\Local\Temp\ipykernel_16008\1726476486.py:161: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_filtrado = df_filtrado[df[col_genero] == genero]
